In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt



def detect_yellow_parasites(image_path,
                            min_area=300,
                            max_area=5000):
    """
    Detect yellow-tinted parasites and return their coordinates.

    Returns:
        boxes: list of bounding boxes (x, y, w, h)
        centroids: list of (cx, cy)
        mask: final binary mask
    """

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image not found")

    # ---- Convert to HSV ----
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ---- Yellow mask ----
    lower_yellow = np.array([15, 60, 50])
    upper_yellow = np.array([35, 255, 200])

    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    # ---- Morphological cleanup ----
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # ---- Find contours ----
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    centroids = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        # Shape filter: elongated objects
        aspect_ratio = max(w, h) / (min(w, h) + 1e-5)
        if aspect_ratio < 1.5:
            continue

        cx = x + w // 2
        cy = y + h // 2

        boxes.append((x, y, w, h))
        centroids.append((cx, cy))

    return boxes, centroids, mask


In [4]:
image_path="2024-03-12_01-25-02-000674 (Edited).jpg"


In [ ]:
img = cv2.imread(image_path)
boxes, centers, mask = detect_yellow_parasites(image_path)

for (x, y, w, h), (cx, cy) in zip(boxes, centers):
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.circle(img, (cx, cy), 3, (0, 0, 255), -1)


plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
# cv2.waitKey(0)
# cv2.destroyAllWindows()


error: OpenCV(4.10.0) /io/opencv/modules/highgui/src/window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'
